# Step 1: Data Loading

Risk-Adjusted Return Prediction


In [1]:
# Setup - add src to path so we can import our modules
import sys
import os

# Go up from Notebooks to project root, then into Src
sys.path.insert(0, os.path.abspath(os.path.join('..', 'Src')))

# Our data loading module
from data.load_data import (
    load_stock_prices,
    load_market_index, 
    load_risk_free_rate,
    check_date_range,
    check_missing_values,
    check_price_columns,
    check_risk_free_rate_units,
    check_date_alignment,
    check_tickers,
    check_survivorship_bias,
    check_missing_trading_days,
    get_trading_dates
)

import pandas as pd

# Data is in ../Data/Raw relative to Notebooks
DATA_DIR = os.path.join('..', 'Data', 'Raw')

---
## 1. Load Raw Datasets

In [2]:
# Load all three datasets
stocks = load_stock_prices(DATA_DIR)
market = load_market_index(DATA_DIR)
rf = load_risk_free_rate(DATA_DIR)

print("Datasets loaded successfully!")
print(f"Stock prices: {len(stocks):,} rows")
print(f"Market index: {len(market):,} rows")
print(f"Risk-free rate: {len(rf):,} rows")

Datasets loaded successfully!
Stock prices: 8,820 rows
Market index: 1,764 rows
Risk-free rate: 1,831 rows


In [3]:
# Quick look at each dataset
print("=" * 50)
print("STOCK PRICES (first 5 rows)")
print("=" * 50)
stocks.head()

STOCK PRICES (first 5 rows)


,date,ticker,open,high,low,close,adj_close,volume
0,2019-01-02,AAPL,38.722500,39.712502,38.557499,39.480000,37.538826,148158800
1,2019-01-03,AAPL,35.994999,36.430000,35.500000,35.547501,33.799690,365248800
2,2019-01-04,AAPL,36.132500,37.137501,35.950001,37.064999,35.242554,234428400
3,2019-01-07,AAPL,37.174999,37.207500,36.474998,36.982498,35.164120,219111200
4,2019-01-08,AAPL,37.389999,37.955002,37.130001,37.687500,35.834450,164101200


In [4]:
print("=" * 50)
print("MARKET INDEX (S&P 500)")
print("=" * 50)
market.head()

MARKET INDEX (S&P 500)


,date,close,adj_close
0,2019-01-02,2510.030029,2510.030029
1,2019-01-03,2447.889893,2447.889893
2,2019-01-04,2531.939941,2531.939941
3,2019-01-07,2549.689941,2549.689941
4,2019-01-08,2574.409912,2574.409912


In [5]:
print("=" * 50)
print("RISK-FREE RATE (3-Month T-Bill)")
print("=" * 50)
rf.head()

RISK-FREE RATE (3-Month T-Bill)


,date,risk_free_rate
0,2019-01-01,NaN
1,2019-01-02,2.37
2,2019-01-03,2.36
3,2019-01-04,2.37
4,2019-01-07,2.41


---
## 2. Date Range Analysis

In [6]:
# Check date ranges for each dataset
check_date_range(stocks, "Stock Prices")
check_date_range(market, "Market Index")
check_date_range(rf, "Risk-Free Rate")

[Stock Prices]
  Start: 2019-01-02
  End:   2026-01-07
  Trading days: 1764

[Market Index]
  Start: 2019-01-02
  End:   2026-01-07
  Trading days: 1764

[Risk-Free Rate]
  Start: 2019-01-01
  End:   2026-01-06
  Trading days: 1831



(Timestamp('2019-01-01 00:00:00'), Timestamp('2026-01-06 00:00:00'))

---
## 3. Missing Values Analysis

In [7]:
check_missing_values(stocks, "Stock Prices")
check_missing_values(market, "Market Index")
check_missing_values(rf, "Risk-Free Rate")

[Stock Prices] Missing Values:
  None - data is complete

[Market Index] Missing Values:
  None - data is complete

[Risk-Free Rate] Missing Values:
  risk_free_rate: 78 (4.26%)



**Note:** Risk-free rate has missing values on holidays (FRED doesn't publish on bank holidays).  
This is expected behavior - will need to forward-fill in Step 2.

---
## 4. Price Column 



In [8]:
check_price_columns(stocks, "Stock Prices")
check_price_columns(market, "Market Index")

[Stock Prices] Price Column Check:
  Close and Adj Close DIFFER (max diff: 11.6058)
  -> Corporate actions (splits/dividends) are present
  -> MUST use Adj Close for return calculations

[Market Index] Close and Adj Close are identical



---
## 5. Risk-Free Rate Unit Check

```

In [9]:
check_risk_free_rate_units(rf)

[Risk-Free Rate Unit Check]
  Min: -0.0500
  Max: 5.3600
  Mean: 2.6544
  -> Appears to be ANNUALIZED PERCENTAGE (DTB3 standard)
  -> To get daily rate: divide by 252 and by 100



---
## 6. Date Alignment Check


In [10]:
common_dates, stock_dates, market_dates, rf_dates = check_date_alignment(
    stocks, market, rf
)

[Date Alignment Check]
  Stock trading days: 1764
  Market trading days: 1764
  Risk-free rate days: 1831
  Common to all: 1763

  [INFO] 68 dates only in rf_rate (includes weekends/holidays)


In [11]:
# Show some dates that are in stocks but not in risk-free
stocks_only = stock_dates - rf_dates
if stocks_only:
    sample = sorted(stocks_only)[:5]
    print("Sample dates in stocks but not in rf_rate:")
    for d in sample:
        print(f"  {d}")

Sample dates in stocks but not in rf_rate:
  2026-01-07


---
## 7. Ticker Coverage

In [12]:
tickers = check_tickers(stocks)

[Ticker Coverage]
  Total tickers: 5
  Tickers: ['AAPL', 'AMZN', 'GOOG', 'META', 'MSFT']

  AAPL: 2019-01-02 to 2026-01-07 (1764 days)
  AMZN: 2019-01-02 to 2026-01-07 (1764 days)
  GOOG: 2019-01-02 to 2026-01-07 (1764 days)
  META: 2019-01-02 to 2026-01-07 (1764 days)
  MSFT: 2019-01-02 to 2026-01-07 (1764 days)



---
## 8. Bias Warning

In [13]:
check_survivorship_bias(stocks, tickers)

[Survivorship Bias Warning]
  Current dataset includes only: AAPL, AMZN, GOOG, META, MSFT
  These are all current market leaders that survived.
  [CAUTION] Delisted/failed stocks are NOT included.
  -> Backtest results may be optimistically biased



---
## 9. Trading Day Gap Analysis

In [14]:
check_missing_trading_days(stocks)

[Trading Day Gaps]
  No suspicious gaps found



In [15]:
print("\n" + "=" * 60)
print("STEP 1 COMPLETE")
print("Data Ingestion & Integrity Verification: PASSED")
print("=" * 60)


STEP 1 COMPLETE
Data Ingestion & Integrity Verification: PASSED
